<a href="https://colab.research.google.com/github/nicoavilan/Intro-Comp-Cuant-UNarino-2026/blob/main/Sesion3_Algoritmos_Aplicaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción a la Computación Cuántica

**Universidad de Nariño — 2026**

Professor: [Nicolás Avilán Vargas](http://www.linkedin.com/in/nicoavilanv)

nicolasg.avilan@urosario.edu.co

**Universidad del Rosario - Escuela de Ciencias e Ingeniería**

**Matemáticas Aplicadas y Ciencias de la Computación**

## Sesión 3 — Algoritmos cuánticos y aplicaciones

### Objetivos de la sesión
Al finalizar esta sesión el participante podrá:
1. Explicar el mecanismo de interferencia constructiva/destructiva como fuente de ventaja cuántica.
2. Implementar de principio a fin el algoritmo de Bernstein-Vazirani.
3. Construir y ejecutar un QAOA simple para un problema de optimización combinatoria (Max-Cut).
4. Describir el panorama de aplicaciones (Shor, QKD/BB84, QML) y las limitaciones actuales (era NISQ).

**Prerrequisitos:** contenidos de las Sesiones 1 y 2 (qubit, medición, entrelazamiento, compuertas paramétricas, transpilación).

---
## 0. Preparación del entorno

In [ ]:
# Instalación (puede tardar 1-2 minutos en Colab)
%pip install qiskit qiskit-aer qiskit[visualization] --quiet

Esta guía funciona con la versión 2.5.2 de qiskit

In [ ]:
import qiskit
print(qiskit.__version__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, array_to_latex
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

%matplotlib inline
print("Entorno listo.")

---
## 1. Interferencia como recurso computacional

### 1.1 La idea central

La ventaja de los algoritmos cuánticos frente a los clásicos no proviene de "probar todo en paralelo y leer la respuesta" (la medición colapsa a un solo resultado, con probabilidades dadas por la regla de Born). Proviene de diseñar circuitos donde:

- Las amplitudes correspondientes a **respuestas incorrectas interfieren destructivamente** (se cancelan),
- Las amplitudes correspondientes a la **respuesta correcta interfieren constructivamente** (se refuerzan),

de modo que, al medir, la respuesta útil aparece con alta probabilidad.

### 1.2 Oráculos cuánticos

Muchos algoritmos (Deutsch-Jozsa, Bernstein-Vazirani, Grover, Shor) usan un **oráculo**: una caja negra unitaria $U_f$ que codifica una función clásica $f:\{0,1\}^n\to\{0,1\}$ mediante

$$
U_f: |x\rangle|y\rangle \mapsto |x\rangle|y\oplus f(x)\rangle.
$$

El oráculo es reversible por construcción (a diferencia de evaluar $f$ clásicamente), lo cual es indispensable porque toda compuerta cuántica debe ser unitaria.

---
## 2. Algoritmo de Bernstein-Vazirani

### 2.1 El problema

Dada una función oculta $f(x) = a\cdot x \pmod 2 = a_0x_0\oplus a_1x_1\oplus\cdots\oplus a_{n-1}x_{n-1}$, para algún $a\in\{0,1\}^n$ desconocido, el objetivo es **encontrar $a$**.

- **Clásicamente:** se requieren $n$ consultas a $f$ (una por cada bit de $a$, evaluando $f(e_i)$ con $e_i$ el vector canónico).
- **Cuánticamente:** basta **una sola consulta** al oráculo $U_f$.

### 2.2 El circuito

1. Preparar $n$ qubits de entrada en $|0\rangle^{\otimes n}$ y 1 qubit auxiliar (ancilla) en $|1\rangle$.
2. Aplicar $H^{\otimes n}$ a la entrada y $H$ al auxiliar (el auxiliar queda en $|-\rangle$).
3. Aplicar el oráculo $U_f$ (implementado como CNOTs desde cada $x_i$ con $a_i=1$ hacia el auxiliar — truco de la **inversión de fase**: con el auxiliar en $|-\rangle$, $U_f$ efectivamente aplica una fase $(-1)^{f(x)}$ sobre $|x\rangle$).
4. Aplicar $H^{\otimes n}$ de nuevo a la entrada.
5. Medir los $n$ qubits de entrada: el resultado es exactamente $a$, con probabilidad 1 (caso ideal, sin ruido).

La clave matemática es que $H^{\otimes n}|x\rangle = \frac{1}{\sqrt{2^n}}\sum_z (-1)^{x\cdot z}|z\rangle$, y al aplicar el oráculo con fase y luego $H^{\otimes n}$ nuevamente, la interferencia deja sobreviviendo únicamente el término $|z=a\rangle$.

Cabe observar que los estados

$$|+\rangle = \frac{1}{\sqrt{2}}\left(|0\rangle +|1\rangle\right),\qquad |-\rangle = \frac{1}{\sqrt{2}}\left(|0\rangle -|1\rangle\right)$$
se obtienen de
$$|+\rangle = H|0\rangle ,\qquad |-\rangle = H|1\rangle $$
y son estados propios de $X$
$$X|+\rangle = |+\rangle ,\qquad X|-\rangle = -|-\rangle $$

El circuito determina $a$ aplicando el oráculo una sola vez sobre una superposición de todos los $x$, de modo que $f(x)$ queda codificado como una fase imperceptible en cada rama — y es la interferencia posterior lo que convierte esa información de fase en el valor medible de $a$.

In [ ]:
def oraculo_bv(a_str): # (Bernstein-Vazirani)
    """Construye el oraculo de fase para f(x) = a.x mod 2, dado a como string de bits, ej '1011'."""
    n = len(a_str)
    oraculo = QuantumCircuit(n + 1, name=f"Oraculo_a={a_str}")
    for i, bit in enumerate(reversed(a_str)):  # convencion: qubit 0 = bit menos significativo
        if bit == '1':
            oraculo.cx(i, n)  # CNOT desde x_i hacia el auxiliar (qubit n)
    return oraculo

def circuito_bernstein_vazirani(a_str):
    n = len(a_str)
    qc = QuantumCircuit(n + 1, n)

    # Paso 1: auxiliar en |1>
    qc.x(n)
    qc.barrier()

    # Paso 2: H en todos
    qc.h(range(n + 1))
    qc.barrier()

    # Paso 3: oraculo
    #qc.compose(oraculo_bv(a_str), inplace=True)
    qc.append(oraculo_bv(a_str), range(n + 1))
    qc.barrier()

    # Paso 4: H en la entrada
    qc.h(range(n))
    qc.barrier()

    # Paso 5: medicion de la entrada
    qc.measure(range(n), range(n))
    return qc


In [ ]:
a_secreto = "1011"
qc_bv = circuito_bernstein_vazirani(a_secreto)
display(qc_bv.draw(output='mpl'))

In [ ]:
sim = AerSimulator()
qc_t = transpile(qc_bv, sim)
counts = sim.run(qc_t, shots=2000).result().get_counts()
print("Resultados:", counts)
print("Valor secreto a =", a_secreto)

plot_histogram(counts, title=f"Bernstein-Vazirani: se espera medir '{a_secreto}' con probabilidad 1")


**Observa:** con una sola ejecución del oráculo (y sin ruido), el resultado medido coincide exactamente con $a$ — cero incertidumbre, a diferencia de las $n$ consultas necesarias clásicamente. Esta es una separación de complejidad de consultas *probada*, aunque para un problema construido específicamente para lucir esta ventaja (no es representativo de utilidad práctica general).

---
## 3. QAOA: optimización combinatoria variacional
(Quantum approximate optimization algorithm)

### 3.1 El problema Max-Cut

Dado un grafo $G=(V,E)$, **Max-Cut** busca una partición de $V$ en dos conjuntos que maximice el número de aristas "cortadas" (con un extremo en cada conjunto). Es un problema NP-difícil en general, y sirve como caso de prueba estándar para algoritmos de optimización cuántica.

![Problema max-cut.](https://quantum.cloud.ibm.com/learning/images/courses/quantum-computing-in-practice/utility-scale-qaoa/maxcut-illustration.avif?dpl=0-1-1564-809b8beb4bbb)
*Fuente: IBM Quantum Learning — [Utility-scale QAOA
](https://quantum.cloud.ibm.com/learning/en/courses/quantum-computing-in-practice/utility-scale-qaoa)*

Codificamos cada nodo $i$ con un qubit: $z_i\in\{0,1\}$ (o $\{-1,+1\}$) indica a qué conjunto pertenece. El costo a maximizar es

$$
C(z) = \sum_{(i,j)\in E} \frac{1-z_iz_j}{2}, \qquad z_i \in \{-1,+1\}.
$$
observando que
$$
\frac{1-z_iz_j}{2} = \begin{cases}0\quad z_i= z_j\quad (\text{Están en el mismo conjunto})\\ 1\quad z_i\not= z_j\quad (\text{Están en conjuntos diferentes}) \end{cases}
$$
Este costo se traduce a un **Hamiltoniano de costo** $H_C = \sum_{(i,j)\in E}\frac{1}{2}(\mathbb{1}-Z_iZ_j)$, diagonal en la base computacional, tal que $H_C|z\rangle = C(z)|z\rangle$.

En este punto se tiene en cuenta que los estados de la base computacional $\{|0\rangle, |1\rangle\}$ son vectores propios del operador $Z$. Esto es, $Z|0\rangle = |0\rangle$ y $Z|1\rangle = -|1\rangle$.

### 3.2 El ansatz QAOA

QAOA (*Quantum Approximate Optimization Algorithm*) construye el estado ansatz alternando $p$ **capas**, cada una formada por dos operadores unitarios parametrizados — uno asociado al costo y otro al "mezclador":

$$
|\gamma,\beta\rangle = \underbrace{e^{-i\beta_p H_B}e^{-i\gamma_p H_C}}_{\text{capa } p}\cdots\underbrace{e^{-i\beta_1 H_B}e^{-i\gamma_1 H_C}}_{\text{capa } 1}\; H^{\otimes n}|0\rangle^{\otimes n}
$$

donde:
- $p$ es el **número de capas** del ansatz (un hiperparámetro que tú eliges antes de optimizar; más capas → ansatz más expresivo, pero circuito más profundo y más sensible al ruido).
- $\gamma=(\gamma_1,\dots,\gamma_p)$ y $\beta=(\beta_1,\dots,\beta_p)$ son los **ángulos de rotación** de cada capa — los $2p$ parámetros reales que el bucle clásico va a ajustar. $\gamma_k,\beta_k$ denotan específicamente los ángulos de la capa $k$-ésima.
- $H_C$ es el Hamiltoniano de costo (codifica el problema de optimización — Max-Cut en nuestro caso).
- $H_B=\sum_i X_i$ es el Hamiltoniano "mezclador": aplica el operador de Pauli $X$ a **cada** qubit $i$ del sistema. Su rol es generar exploración — evita que el estado quede atrapado en los estados base de $H_C$, permitiendo que la optimización se mueva entre distintas configuraciones candidatas.

Los ángulos $(\gamma,\beta)$ se **optimizan clásicamente** (bucle híbrido cuántico-clásico): el circuito cuántico evalúa, para unos ángulos dados, el valor esperado del costo

$$
\langle\gamma,\beta|H_C|\gamma,\beta\rangle
$$

y un optimizador clásico (p. ej. COBYLA) ajusta $(\gamma,\beta)$ iterativamente para maximizar ese valor esperado.

In [ ]:
# Grafo pequeño para Max-Cut (5 nodos)
G = nx.Graph()
G.add_edges_from([(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)])

fig, ax = plt.subplots(figsize=(5,4))
pos = nx.spring_layout(G, seed = 42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=600, ax=ax)
ax.set_title("Grafo para Max-Cut")
plt.show()

print("Nodos:", G.nodes())
print("Aristas:", G.edges())


In [ ]:
from qiskit.quantum_info import SparsePauliOp

def hamiltoniano_costo_maxcut(G):
    """Construye H_C = sum_{(i,j) in E} 0.5*(I - Z_i Z_j) como SparsePauliOp."""
    n = G.number_of_nodes()
    pauli_list = []
    for (i, j) in G.edges():
        z_string = ['I'] * n
        z_string[i] = 'Z'
        z_string[j] = 'Z'
        pauli_list.append((''.join(reversed(z_string)), -0.5))  # -0.5 Z_i Z_j
    # termino constante 0.5*|E|
    const = 0.5 * G.number_of_edges()
    pauli_list.append(('I'*n, const))
    return SparsePauliOp.from_list(pauli_list)

H_C = hamiltoniano_costo_maxcut(G)
print(H_C)


In [ ]:
# Ansatz QAOA con p=1 capa, usando la libreria de Qiskit
p_capas = 1
ansatz = QAOAAnsatz(cost_operator=H_C, reps=p_capas)
ansatz = ansatz.decompose(reps=2)
print(f"Numero de parametros: {ansatz.num_parameters}")
display(ansatz.draw(output='mpl', fold=120))


El siguiente bloque corresponde al dicho el diagrama "cuántico evalúa, clásico ajusta, repite" que define QAOA.

In [ ]:
#StatevectorEstimator() ejecuta el circuito y da valores esperados de uno o más operadores
estimator = StatevectorEstimator()

def costo_esperado(params):
    """Evalua -<H_C> en el ansatz ligado a 'params' (negativo porque scipy minimiza)."""
    pub = (ansatz, [H_C], [params])
    resultado_est = estimator.run([pub]).result()
    valor_esperado = resultado_est[0].data.evs[0]
    return -valor_esperado
# minimiza (- valor esperado)

np.random.seed(42)
x0 = np.random.uniform(0, 2*np.pi, ansatz.num_parameters)

resultado_opt = minimize(costo_esperado, x0, method="COBYLA", options={"maxiter": 100})

print("Angulos optimos encontrados (gamma, beta):", resultado_opt.x)
print("Corte maximo estimado:", -resultado_opt.fun)


In [ ]:
# Muestreamos el circuito con los angulos optimos para obtener la particion propuesta
ansatz_medido = ansatz.copy()
ansatz_medido.measure_all()

sampler = StatevectorSampler()
pub = (ansatz_medido, resultado_opt.x)
resultado_sample = sampler.run([pub], shots=4000).result()
counts = resultado_sample[0].data.meas.get_counts()

top_estados = sorted(counts.items(), key=lambda kv: -kv[1])[:5]
total_shots = sum(counts.values())
print("Bitstrings mas probables (particion propuesta):")
for bitstring, n in top_estados:
    print(f"  {bitstring}: probabilidad {n/total_shots:.3f}")

mejor_bitstring = top_estados[0][0]
print(f"\nMejor particion encontrada: {mejor_bitstring}")

# Visualizacion de la particion sobre el grafo
colores = ['red' if bit == '1' else 'lightblue' for bit in reversed(mejor_bitstring)]
fig, ax = plt.subplots(figsize=(5,4))
nx.draw(G, pos, with_labels=True, node_color=colores, node_size=600, ax=ax)
ax.set_title(f"Particion Max-Cut propuesta: {mejor_bitstring}")
plt.show()


**Observa:** con $p=1$ capa QAOA suele dar una aproximación razonable pero no necesariamente óptima; aumentar $p$ (a costa de circuitos más profundos, más sensibles al ruido) generalmente mejora la aproximación. Esta tensión -más expresividad vs. más ruido acumulado— es central en la era NISQ.

---
## 4. Panorama de aplicaciones y desafíos actuales

### 4.1 Criptografía: algoritmo de Shor

El **algoritmo de Shor** factoriza enteros en tiempo polinomial en $\log N$ (vs. tiempo subexponencial para el mejor algoritmo clásico conocido), usando como subrutina la **transformada cuántica de Fourier** para encontrar el período de una función modular. Esto compromete la seguridad de RSA y otros criptosistemas de clave pública basados en la dificultad de factorización o logaritmo discreto — de ahí el interés en **criptografía poscuántica** (basada en retículos, códigos, etc.).

### 4.2 Distribución cuántica de claves: BB84

Como contraparte **defensiva**, el protocolo **BB84** usa las propiedades cuánticas de la medición (en particular, que medir en una base incorrecta perturba el estado de forma detectable) para permitir que dos partes establezcan una clave secreta compartida con **seguridad garantizada por las leyes de la física**, no por dificultad computacional: cualquier intento de espionaje (Eve midiendo los qubits en tránsito) introduce errores detectables en la clave resultante.

### 4.3 Aprendizaje automático cuántico (QML)

Las técnicas de QML codifican datos clásicos en estados cuánticos (*data encoding*, p. ej. mediante compuertas $R_y(\theta_i)$ con $\theta_i$ proporcional a cada característica) y usan circuitos variacionales entrenables (similares al ansatz QAOA) como **clasificadores cuánticos** (VQC) o para estimar kernels cuánticos. La promesa es explotar espacios de características de alta dimensión de forma eficiente; la evidencia de ventaja práctica sobre métodos clásicos sigue siendo un área de investigación activa, sin consenso general.

### 4.4 Era NISQ y limitaciones actuales

*Noisy Intermediate-Scale Quantum* (NISQ) describe el estado actual del hardware: decenas a cientos de qubits físicos, sin corrección de errores completa, con tasas de error por compuerta del orden de $10^{-3}$–$10^{-2}$. Esto limita la profundidad práctica de los circuitos ejecutables antes de que el ruido destruya la señal útil (como viste en la Sesión 2). La **corrección de errores cuánticos** (códigos de superficie, entre otros) es la vía hacia la computación cuántica **tolerante a fallos**, pero requiere una sobrecarga significativa de qubits físicos por qubit lógico — un horizonte activo de investigación, no una realidad de despliegue actual.

In [ ]:
# Tabla resumen comparativa (solo con fines ilustrativos, valores aproximados de orden de magnitud)
import pandas as pd

resumen = pd.DataFrame({
    "Algoritmo/Protocolo": ["Bernstein-Vazirani", "Deutsch-Jozsa", "Shor (factorizacion)",
                             "Grover (busqueda)", "QAOA", "VQE", "BB84 (QKD)"],
    "Tipo de ventaja": ["Consultas exactas", "Consultas exactas", "Exponencial (conjeturada)",
                         "Cuadratica (probada)", "Heuristica/variacional", "Heuristica/variacional",
                         "Seguridad fisica, no velocidad"],
    "Requiere hardware tolerante a fallos": ["No", "No", "Si (para instancias utiles)",
                                              "Parcial", "No (NISQ-friendly)", "No (NISQ-friendly)", "No"]
})
resumen


---
## 5. Resumen de la sesión

| Concepto | Idea clave |
|---|---|
| Interferencia | Fuente de ventaja cuántica: amplitudes correctas se refuerzan, incorrectas se cancelan |
| Oráculo | Unitario reversible que codifica una función clásica |
| Bernstein-Vazirani | Encuentra $a$ oculto en $f(x)=a\cdot x$ con 1 consulta (vs. $n$ clásicas) |
| Deutsch-Jozsa | Distingue $f$ constante vs. balanceada en 1 consulta |
| QAOA | Ansatz variacional híbrido para optimización combinatoria (Max-Cut como ejemplo) |
| Shor / BB84 | Amenaza y defensa criptográfica basadas en principios cuánticos distintos |
| QML | Codificación de datos + circuitos variacionales entrenables |
| NISQ | Régimen actual: qubits ruidosos, sin corrección de errores completa |

Con esto se cierran los tres bloques del curso: fundamentos (Sesión 1), construcción/simulación de circuitos (Sesión 2), y algoritmos/aplicaciones (Sesión 3).

---
## 6. Ejercicios propuestos

### Ejercicio 1 — Bernstein-Vazirani con distintos valores de $a$
Ejecuta `circuito_bernstein_vazirani` para al menos 4 valores distintos de `a_secreto` (incluyendo el caso `a = "0000"`). Verifica en cada caso que el resultado medido coincide con `a` (cuidado con el orden de bits de Qiskit — little-endian). ¿Qué circuito resulta cuando $a=0$?

In [ ]:
# Ejercicio 1 - tu codigo aqui
for a_prueba in ["0000", "0001", "1100", "1111"]:
    qc = circuito_bernstein_vazirani(a_prueba)
    # ejecuta y verifica




### Ejercicio 2 — Deutsch-Jozsa
Implementa el algoritmo de Deutsch-Jozsa: construye dos oráculos, uno para una función **constante** ($f(x)=0\ \forall x$) y otro para una función **balanceada** (por ejemplo, $f(x) = x_0$, es decir, depende solo del primer bit). Ejecuta el circuito completo (idéntico en estructura al de Bernstein-Vazirani) para cada oráculo y confirma que obtienes $|0\rangle^{\otimes n}$ en el caso constante y cualquier otro resultado en el caso balanceado.

In [ ]:
# Ejercicio 2 - tu codigo aqui

def oraculo_constante(n):
    # f(x) = 0 para todo x -> oraculo es el circuito identidad (no hace nada)
    return QuantumCircuit(n + 1, name="Oraculo_constante")

def oraculo_balanceada(n):
    # f(x) = x_0 -> CNOT desde qubit 0 hacia el auxiliar
    oraculo = QuantumCircuit(n + 1, name="Oraculo_balanceada")
    oraculo.cx(0, n)
    return oraculo

# Reutiliza la estructura de circuito_bernstein_vazirani, sustituyendo el oraculo



### Ejercicio 3 — QAOA en otro grafo
Repite el flujo de QAOA (Sección 3) para un grafo distinto: un ciclo de 4 nodos $C_4$ (aristas $(0,1),(1,2),(2,3),(3,0)$). El corte máximo teórico de $C_4$ es 4 (grafo bipartito perfecto). Compara el corte encontrado por QAOA con $p=1$ contra el óptimo conocido. Luego repite con $p=2$ capas y observa si mejora la aproximación.

In [ ]:
# Ejercicio 3 - tu codigo aqui
G_ciclo = nx.Graph()
G_ciclo.add_edges_from([(0,1),(1,2),(2,3),(3,0)])

# repite: hamiltoniano_costo_maxcut(G_ciclo), QAOAAnsatz(cost_operator=..., reps=p),
# y el mismo bucle de minimize(costo_esperado, ...) de la Seccion 3.
# Compara el resultado con p=1 vs p=2 capas.



### Ejercicio 4 — Fuerza bruta vs. QAOA
Para el grafo de 5 nodos de la Sección 3, calcula el corte máximo **exacto** por fuerza bruta (evaluando las $2^5=32$ particiones posibles con `C(z)` definido en la Sección 3.1). Compara el valor óptimo exacto contra el valor que encontró QAOA. ¿Qué tan cerca estuvo la aproximación variacional?

In [ ]:
# Ejercicio 4 - tu codigo aqui
def costo_particion(G, bitstring):
    z = [1 if b == '0' else -1 for b in bitstring]
    return sum(0.5*(1 - z[i]*z[j]) for (i,j) in G.edges())

# Itera sobre las 2^5 combinaciones de bitstrings y encuentra el maximo




### Ejercicio 5 — Reto (opcional): BB84 simplificado
Implementa una simulación simplificada de BB84 entre Alice y Bob (sin espía) para una clave de 10 bits:
1. Alice genera 10 bits aleatorios y 10 bases aleatorias (`Z` o `X`).
2. Alice prepara cada qubit según su bit y base (usa $H$ para la base $X$).
3. Bob elige 10 bases aleatorias e mide cada qubit en su base elegida.
4. Alice y Bob comparan (clásicamente) sus bases: descartan los qubits donde no coincidieron, y el resto forma la clave compartida.

Verifica que, cuando las bases de Alice y Bob coinciden, el bit medido por Bob siempre coincide con el bit original de Alice (en ausencia de ruido/espía).

In [ ]:
# Ejercicio 5 (reto) - tu codigo aqui
import random
random.seed(1)

n_bits = 10
bits_alice = [random.randint(0,1) for _ in range(n_bits)]
bases_alice = [random.choice(['Z','X']) for _ in range(n_bits)]
bases_bob = [random.choice(['Z','X']) for _ in range(n_bits)]

# Para cada i: construye un circuito de 1 qubit, prepara segun (bits_alice[i], bases_alice[i]),
# mide en bases_bob[i], y compara con bits_alice[i] cuando bases_alice[i]==bases_bob[i]



---
## Cierre del curso

Con esta sesión concluyen los tres bloques del curso *Introducción a la Computación Cuántica*. Has recorrido el camino desde el postulado de superposición de un único qubit hasta la implementación de algoritmos cuánticos con ventaja demostrable y heurísticas variacionales para optimización — usando Qiskit como herramienta de construcción, simulación y análisis en cada paso.

## Referencias para profundizar
- Nielsen, M. A. & Chuang, I. L. *Quantum Computation and Quantum Information*, caps. 1.4 (Deutsch-Jozsa), 5 (transformada de Fourier cuántica y Shor), 12 (criptografía cuántica).
- Farhi, E., Goldstone, J., Gutmann, S. (2014). *A Quantum Approximate Optimization Algorithm*. arXiv:1411.4028.
- Bennett, C. H. & Brassard, G. (1984). *Quantum cryptography: Public key distribution and coin tossing* (protocolo BB84).
- Qiskit Optimization — documentación: https://qiskit-community.github.io/qiskit-optimization/
- IBM Quantum Learning — módulos de QAOA y algoritmos de consulta: https://learning.quantum.ibm.com/


**Para** informar de errores o hacer sugerencias: nicolasg.avilan@urosario.edu.co